# Practice 24 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Basics

### Task 1: diamonds — profile by cut and clarity

Load `sns.load_dataset('diamonds')`. Set a `(cut, clarity)` MultiIndex and sort it.

Answer these five questions — no step-by-step hints, figure out the right approach yourself:

- How many distinct `(cut, clarity)` combinations exist in the data?
- Which `clarity` grade appears in the most `cut` categories?
- Among `'Premium'` cut diamonds only, which `clarity` has the highest median `price`?
- Get all `'VS1'` clarity rows across every cut type. What fraction have `carat > 1.0`? Use NumPy.
- Compute the Pearson correlation between `carat` and `price` for each `cut`. Which cut shows the strongest relationship? Use NumPy.

In [120]:
# Your code here
diamonds = sns.load_dataset('diamonds')
diamonds = diamonds.set_index(['cut', 'clarity']).sort_index()
diamonds.index.unique().shape
diamonds

cla_len={}
for cla in diamonds.index.get_level_values('clarity').unique():
    cla_len[cla] = len(diamonds.xs(cla,level='clarity').index.get_level_values('cut').unique())

cla_len

diamonds.groupby(level='clarity')['price'].apply(lambda x: x.index.get_level_values('cut').nunique()) 

#diamonds.xs('Premium', level='cut').groupby(level = 'clarity')['price'].mean().idxmax()

#((diamonds.xs('VS1', level='clarity')['carat']>1).sum())/(diamonds.xs('VS1', level='clarity').shape[0])
#cor = diamonds.groupby('cut').apply(lambda x: np.corrcoef(x['carat'], x['price'])[0,1])
#
# 
# cor.idxmax()



/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_38875/1720154027.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  diamonds.groupby(level='clarity')['price'].apply(lambda x: x.index.get_level_values('cut').nunique())


clarity
IF      5
VVS1    5
VVS2    5
VS1     5
VS2     5
SI1     5
SI2     5
I1      5
Name: price, dtype: int64

---
## Level 2 — Transformations

### Task 2: Sales reps — pipeline then reshape

Write all three functions from scratch, then chain them with `.pipe()`:

- `add_revenue(df)`: `revenue = units * unit_price`
- `normalize_by_region(df)`: add `revenue_pct_of_region_max` — each row's revenue as a fraction of the maximum revenue in its region (use `transform`)
- `flag_top_performer(df)`: add `top_performer` — `True` if a row's revenue exceeds that rep's own mean revenue across all months (use `transform`)

After the pipeline:
- Reshape to wide format: reps as rows, months as columns, `revenue` as values. Store as `sales_wide`.
- Which rep has the highest variance in monthly revenue? Use NumPy.
- Use `.xs()` to pull out the `'Mar'` data from `sales_wide`.

In [124]:
# Starter data — don't change this
np.random.seed(7)
reps     = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank']
regions  = ['East', 'East', 'West', 'West', 'Central', 'Central']
months   = ['Jan', 'Feb', 'Mar', 'Apr']

sales = pd.DataFrame({
    'rep':        np.repeat(reps, len(months)),
    'region':     np.repeat(regions, len(months)),
    'month':      months * len(reps),
    'units':      np.random.randint(5, 50, size=len(reps) * len(months)),
    'unit_price': np.random.choice([20, 35, 50, 75], size=len(reps) * len(months)),
})

# Your code here

def add_revenue(df):
    df['revenue'] = df['units']*df['unit_price']
    return df
def normalize_by_region(df):
    df['revenue_pct_of_region_max'] = df['revenue']/(df.groupby('region')['revenue'].transform('max'))
    return df
def flag_top_performer(df):
    df['top_performer'] = df['revenue']>df.groupby('rep')['revenue'].transform('mean') 
    return df


res = (
    sales
    .pipe(add_revenue)
    .pipe(normalize_by_region)
    .pipe(flag_top_performer)
)

sales_wide = res.pivot_table(
    index = 'rep',
    columns = 'month',
    values = 'revenue'
)

v = res.groupby('rep')['revenue'].agg('var')
v.index[np.argmax(v)]

#sales_wide.unstack().xs('Mar', level='month')
sales_wide['Mar']


rep
Alice     160.0
Bob       660.0
Carol    1500.0
Dave     2200.0
Eve      3675.0
Frank    1800.0
Name: Mar, dtype: float64

### Task 3: stack() / unstack() / .xs() — temperature anomalies

A wide DataFrame of monthly temperatures (cities × months) has been created for you.

1. Stack to long format. Name the value column `'temp_c'`. Store as `temps_long`.
2. Add `temp_f`: convert Celsius to Fahrenheit using NumPy (no `.apply()`).
3. Add `anomaly`: each row's `temp_c` minus that city's mean `temp_c` across all months. Use `transform`.
4. Which city has the largest anomaly range (max minus min)? Use NumPy.
5. Use `.xs()` to get all `'Jul'` readings across every city.
6. Unstack back to wide format — `temp_c` values only.

In [62]:
# Starter data — don't change this
np.random.seed(41)
temps = pd.DataFrame({
    'city': ['Oslo', 'Cairo', 'Sydney', 'Toronto', 'Mumbai'],
    'Jan': np.round(np.random.uniform(-10, 10, 5), 1),
    'Apr': np.round(np.random.uniform(5, 25, 5), 1),
    'Jul': np.round(np.random.uniform(15, 40, 5), 1),
    'Oct': np.round(np.random.uniform(0, 30, 5), 1),
}).set_index('city')

# Your code here

temps_long = temps.stack().to_frame('temp_c')
temps_long['temp_f'] = temps_long['temp_c'].apply(lambda x: x*9/5+32) 
temps_long['anomaly'] = temps_long['temp_c']-temps_long.groupby(level='city')['temp_c'].transform('mean')
r = temps_long.groupby('city')['anomaly'].agg('max')-temps_long.groupby('city')['anomaly'].agg('min')
r.index[np.argmax(r)]
temps_long.xs('Jul', level = 1)
w = temps_long['temp_c'].unstack()
w

,Jan,Apr,Jul,Oct
city,,,,
Oslo,-5.0,17.1,23.3,2.1
Cairo,-9.1,8.8,22.1,21.1
Sydney,3.5,18.4,19.7,9.4
Toronto,-9.1,23.3,22.9,22.4
Mumbai,-7.7,13.4,27.0,11.9


---
## Level 3 — Aggregation

### Task 4: Course enrollment — four questions, no recipe

An enrollment DataFrame has been created for you: 8 students, each taking 6 courses across 3 departments.

Four questions. No step-by-step breakdown — figure out the right approach for each:

1. Which department has the highest mean grade across all its enrolled students?
2. Compute `weighted_grade = grade × credits` for each row. Which student has the highest *total* weighted grade?
3. Rank students within each department by their mean grade across that department's courses. Show the top-ranked student per department.
4. Build a pivot: departments as rows, courses as columns, mean grade as values. Stack it, then use `.xs()` to extract the full grade profile for `'Science'`.

In [80]:
# Starter data — don't change this
np.random.seed(13)
student_names = ['Alex', 'Blake', 'Casey', 'Dana', 'Eli', 'Fay', 'Glen', 'Harper']
course_info = [
    ('Calculus',   'Math',       4),
    ('Statistics', 'Math',       3),
    ('Biology',    'Science',    4),
    ('Chemistry',  'Science',    4),
    ('Literature', 'Humanities', 3),
    ('History',    'Humanities', 3),
]
rows = [
    {'student': s, 'course': c, 'dept': d, 'credits': cr,
     'grade': round(np.random.uniform(2.0, 4.0), 2)}
    for s in student_names
    for c, d, cr in course_info
]
enrollments = pd.DataFrame(rows)

# Your code here
enrollments.groupby('dept')['grade'].mean().idxmax()
enrollments['weighted_grade'] = enrollments['grade']*enrollments['credits']
enrollments.groupby('student')['weighted_grade'].sum().idxmax()
enrollments['mean_dept'] = enrollments.groupby(['student','dept'])['grade'].transform('mean')
enrollments['r']= enrollments.groupby('dept')['mean_dept'].rank(ascending=False)

enrollments[enrollments['r']<=5][['student','dept']].drop_duplicates()

p = enrollments.pivot_table(
    index = 'dept',
    columns = 'course',
    values='grade'

).stack()
p.xs('Science', level='dept')

course
Biology      3.31125
Chemistry    3.13750
dtype: float64

---
## Level 4 — Real-world

### Task 5: penguins — open investigation

Load `sns.load_dataset('penguins').dropna().reset_index(drop=True)`.

Four questions. No scaffolding — pick your own methods.

1. Which `(species, island)` pair has the most *internally consistent* body measurements? Define "consistent" yourself (e.g. low coefficient of variation, low std, low range — your call) and show the evidence numerically.
2. Build a `.pipe()` chain with at least two functions of your own design. The pipeline should produce at least two new columns and end with a meaningful summary (groupby, pivot, or rank — your choice).
3. Create a reshaped view — stack, unstack, or pivot — that makes it easy to compare a measurement across both `species` and `island` at once. Use `.xs()` somewhere in the process.
4. What is the strongest pairwise numerical relationship in this dataset? Use `np.corrcoef` to check all pairs and identify the winner.

In [90]:
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,bill_ratio
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,2.090909
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,2.270115
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,2.238889
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,1.901554
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,1.907767
...,...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female,3.445255
329,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,3.272727
330,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,3.210191
331,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,3.054054


In [118]:
# Your code here
penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)
penguins.groupby(['species','island'])['body_mass_g'].agg('std').idxmin()

def bill_ratio(df):
    df['bill_ratio'] = df['bill_length_mm']/df['bill_depth_mm']
    df['billflipper_ratio'] = df['bill_length_mm']/df['flipper_length_mm']

    return df

def pivot_p(df):
    p = df.pivot_table(
        index = 'species',
        columns = 'island',
        values = 'bill_ratio'

    )
    return p


res = (
    penguins
    .pipe(bill_ratio)
    .pipe(pivot_p)
)

penguins.pivot_table(
    index = 'species',
    columns = 'island',
    values = 'bill_ratio'
).stack().xs('Adelie', level = 'species')


n = penguins.select_dtypes(include='number')
c = np.corrcoef(n.T)
np.fill_diagonal(c, np.nan)
np.nanmax(c)

np.float64(0.8729788985653611)

In [102]:
n

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,bill_ratio,billflipper_ratio
0,39.1,18.7,181.0,3750.0,2.090909,0.216022
1,39.5,17.4,186.0,3800.0,2.270115,0.212366
2,40.3,18.0,195.0,3250.0,2.238889,0.206667
3,36.7,19.3,193.0,3450.0,1.901554,0.190155
4,39.3,20.6,190.0,3650.0,1.907767,0.206842
...,...,...,...,...,...,...
328,47.2,13.7,214.0,4925.0,3.445255,0.220561
329,46.8,14.3,215.0,4850.0,3.272727,0.217674
330,50.4,15.7,222.0,5750.0,3.210191,0.227027
331,45.2,14.8,212.0,5200.0,3.054054,0.213208
